# HFSS 2.45 GHz — Reproducible Field Analysis (Gate 1)

This Colab notebook reconstructs the complex electric field exported by HFSS and generates **40 numeric electric-field magnitude matrices**, 40 physical-scale figures, and 40 robust presentation-scale figures.

**Gate 1 intentionally stops before the Shearlet transform.**

In [ ]:
# Colab environment
from pathlib import Path
import csv, hashlib, json, os, subprocess, sys, zipfile, shutil

IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

REPO_URL = "https://github.com/codebycristian-dev/hfss-shearlet-colab.git"
REPOSITORY_IS_PRIVATE = True  # Set False if the repository becomes public.
REPO_ROOT = Path("/content/hfss-shearlet-colab") if IN_COLAB else Path("..").resolve()
if IN_COLAB and not (REPO_ROOT / "src").is_dir():
    clone_env = os.environ.copy()
    askpass_path = Path("/content/codex_git_askpass.sh")
    if REPOSITORY_IS_PRIVATE:
        from google.colab import userdata
        try:
            github_token = userdata.get("GITHUB_TOKEN")
        except Exception as exc:
            raise RuntimeError("Private repository access requires a GITHUB_TOKEN in Colab Secrets.") from exc
        if not github_token:
            raise RuntimeError("Private repository access requires a GITHUB_TOKEN in Colab Secrets.")
        askpass_path.write_text("#!/bin/sh\ncase \"$1\" in\n*Username*) printf '%s\\n' x-access-token ;;\n*Password*) printf '%s\\n' \"$GITHUB_TOKEN\" ;;\nesac\n")
        askpass_path.chmod(0o700)
        clone_env.update({"GIT_ASKPASS": str(askpass_path), "GIT_TERMINAL_PROMPT": "0", "GITHUB_TOKEN": github_token})
        github_token = None
    try:
        result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], env=clone_env, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
        if result.returncode:
            raise RuntimeError("Repository clone failed. Verify GITHUB_TOKEN access or set REPOSITORY_IS_PRIVATE=False for a public repository.")
    finally:
        clone_env.pop("GITHUB_TOKEN", None)
        if askpass_path.exists():
            askpass_path.unlink()
if not (REPO_ROOT / "src").is_dir():
    raise RuntimeError(f"Repository src/ directory not found at {REPO_ROOT}")
sys.path.insert(0, str(REPO_ROOT))
print("Repository:", REPO_ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], check=True)
git_result = subprocess.run(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], capture_output=True, text=True)
GIT_COMMIT_SHA = git_result.stdout.strip() if git_result.returncode == 0 else None
print("Git commit:", GIT_COMMIT_SHA or "unavailable")


## 1. Data source

For development, keep the HFSS database outside GitHub. Preferred option: place the original ZIP in Google Drive.

Set `DATA_SOURCE = "DRIVE"` or `"UPLOAD"` below.

In [ ]:
DATA_SOURCE = "DRIVE"  # "DRIVE" or "UPLOAD"
DATA_ZIP_NAME = "Entregas_papper(1).zip"

if IN_COLAB and DATA_SOURCE == "DRIVE":
    from google.colab import drive
    drive.mount("/content/drive")
    # CHANGE ONLY THIS PATH if needed:
    DATA_ZIP = Path("/content/drive/MyDrive/HFSS_Dataset") / DATA_ZIP_NAME
elif IN_COLAB and DATA_SOURCE == "UPLOAD":
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError("Upload exactly one dataset ZIP.")
    DATA_ZIP = Path("/content") / next(iter(uploaded))
else:
    DATA_ZIP = Path("../data") / DATA_ZIP_NAME

print("Dataset ZIP:", DATA_ZIP)
if not DATA_ZIP.exists():
    raise FileNotFoundError(DATA_ZIP)

dataset_hasher = hashlib.sha256()
with DATA_ZIP.open("rb") as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b""):
        dataset_hasher.update(chunk)
DATASET_SHA256 = dataset_hasher.hexdigest()
print("Dataset filename:", DATA_ZIP.name)
print("Dataset SHA-256:", DATASET_SHA256)


In [ ]:
# Re-extract on every Run All so a previous session cannot supply stale data.
WORK = Path("/content/hfss_work") if IN_COLAB else Path("../.work")
DATA_DIR = WORK / "dataset"
OUTPUT_DIR = WORK / "outputs"

if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
DATA_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(DATA_ZIP) as archive:
    root = DATA_DIR.resolve()
    for member in archive.infolist():
        target = (DATA_DIR / member.filename).resolve()
        if root not in target.parents and target != root:
            raise RuntimeError(f"Unsafe ZIP member path: {member.filename}")
    archive.extractall(DATA_DIR)

print("Extracted dataset to:", DATA_DIR)


## 2. Gate 1 processing

The pipeline reconstructs

\[
\mathbf E = \Re\{\mathbf E\} + j\Im\{\mathbf E\}
\]

and computes

\[
|\mathbf E|=\sqrt{|E_x|^2+|E_y|^2+|E_z|^2}.
\]

The physical quantity is **electric-field magnitude** $|E|$ in V/m, not Poynting or electromagnetic intensity. The geometric mask is applied only after the full numeric field is reconstructed. Mode 1 is `parallel_y`; Mode 2 is `perpendicular_y`. All plots preserve equal physical spacing.

In [ ]:
from src.pipeline import run_intensity_pipeline

rows = run_intensity_pipeline(
    DATA_DIR, OUTPUT_DIR,
    dataset_filename=DATA_ZIP.name,
    dataset_sha256=DATASET_SHA256,
    git_commit_sha=GIT_COMMIT_SHA,
)
print(f"Processed {len(rows)} cuts.")
assert len(rows) == 40, f"Expected 40 cuts, got {len(rows)}"


In [ ]:
# Gate 1 output validation
physical_pngs = sorted((OUTPUT_DIR / "01_intensity" / "physical_shared").rglob("*.png"))
presentation_pngs = sorted((OUTPUT_DIR / "01_intensity" / "presentation_shared").rglob("*.png"))
matrices = sorted((OUTPUT_DIR / "02_numeric").rglob("*.npz"))
metrics = OUTPUT_DIR / "04_metrics" / "field_metrics.csv"
run_metadata_path = OUTPUT_DIR / "04_metrics" / "run_metadata.json"
with metrics.open(newline="", encoding="utf-8") as stream:
    metric_rows = list(csv.DictReader(stream))
run_metadata = json.loads(run_metadata_path.read_text(encoding="utf-8"))
expected_groups = {(f"Mode{mode}", plane) for mode in (1, 2) for plane in ("XZ", "YZ")}
physical_groups = {(path.parent.parent.name, path.parent.name) for path in physical_pngs}
presentation_groups = {(path.parent.parent.name, path.parent.name) for path in presentation_pngs}
matrix_groups = {(path.parent.parent.name, path.parent.name) for path in matrices}

print("Physical shared-scale PNGs:", len(physical_pngs))
print("Presentation shared-scale PNGs:", len(presentation_pngs))
print("Numeric matrices:", len(matrices))
print("Metric rows:", len(metric_rows))
print("Metrics:", metrics)
assert len(physical_pngs) == 40, f"Expected 40 physical PNGs, got {len(physical_pngs)}"
assert len(presentation_pngs) == 40, f"Expected 40 presentation PNGs, got {len(presentation_pngs)}"
assert len(matrices) == 40, f"Expected 40 numeric matrices, got {len(matrices)}"
assert len(metric_rows) == 40, f"Expected 40 metric rows, got {len(metric_rows)}"
assert physical_groups == expected_groups, f"Invalid physical PNG structure: {physical_groups}"
assert presentation_groups == expected_groups, f"Invalid presentation PNG structure: {presentation_groups}"
assert matrix_groups == expected_groups, f"Invalid matrix structure: {matrix_groups}"
assert {row["polarization"] for row in metric_rows} == {"parallel_y", "perpendicular_y"}
assert run_metadata["dataset_filename"] == DATA_ZIP.name
assert run_metadata["dataset_sha256"] == DATASET_SHA256
assert run_metadata["frequency_GHz"] == 2.45
assert run_metadata["quantity"] == "Electric-field magnitude |E|"
assert run_metadata["unit"] == "V/m"
print("GATE 1: PASS")


In [ ]:
# Package Gate 1 outputs
archive_base = str(WORK / "HFSS_Gate1_results_2p45GHz")
zip_path = shutil.make_archive(archive_base, "zip", OUTPUT_DIR)
print("Created:", zip_path)

if IN_COLAB:
    from google.colab import files
    files.download(zip_path)
